# Explore the interim tables

Runs after `mammo flatten`, before `features/build_train_set.py` is written.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd

from mammography.io import read_table

patients = read_table("interim", "patients", parse_dates=["birth_date", "deceased_date"])
patients.shape

## 1. Patient stats

How many patients there are, whether ids are unique, and which columns are
populated enough to be features.

In [ ]:
print(f"rows: {len(patients)}")
print(f"unique patient_id: {patients['patient_id'].nunique()}")
print(f"deceased: {patients['deceased_date'].notna().sum()}")
print("\nnull rate per column:")
print(patients.isna().mean().sort_values(ascending=False).to_string())

### Age and the screening-eligible cohort

Only part of the population is ever invited. The eligible count is the real
sample size for the model — not the row count above.

In [ ]:
today = pd.Timestamp("today").normalize()
patients["age"] = ((today - patients["birth_date"]).dt.days / 365.25).astype(int)

eligible = patients[
    (patients["gender"] == "female")
    & patients["age"].between(45, 69)
    & patients["deceased_date"].isna()
]

print(patients["gender"].value_counts(dropna=False).to_string())
print(f"\nage: min={patients['age'].min()} median={patients['age'].median()} max={patients['age'].max()}")
print(f"eligible (female, 45-69, alive): {len(eligible)} of {len(patients)}")

patients["age"].hist(bins=30)
plt.xlabel("age")
plt.ylabel("patients")
plt.show()

### Geography

Candidate for the `region` feature — check the cardinality and the rare tail
before using it.

In [ ]:
for column in ["state", "district", "city"]:
    counts = patients[column].value_counts(dropna=False)
    print(f"{column}: {counts.size} distinct")
    print(counts.head(10).to_string())
    print()

## 2. Days from invitation to mammography

Decides `attendance_window_days` for the label. Every invitation is paired with
every later appointment of the same patient on purpose — the point is the shape
of the delay distribution, not a matched join. Read the cutoff off the plot:
where the bulk ends, the rest is attendance for unrelated reasons.

Needs the invitation layer, which the Synthea sample data does not contain —
this cell fails until `communications` and `appointments` exist.

In [ ]:
comms = read_table("interim", "communications", parse_dates=["sent_date"])
appts = read_table("interim", "appointments", parse_dates=["start"])

pairs = comms[["patient_id", "sent_date"]].merge(
    appts[["patient_id", "start"]], on="patient_id", how="inner"
)
pairs["days_to_appointment"] = (pairs["start"] - pairs["sent_date"]).dt.days
after = pairs.loc[pairs["days_to_appointment"].between(0, 730), "days_to_appointment"]

after.hist(bins=73)
plt.xlabel("days from invitation to mammography")
plt.ylabel("appointments")
plt.show()

after.describe()